# ApexPINN M2/F6: SageMaker managed training

Prepared, NOT RUN. This notebook controls a separate L4 training job; it never fits locally.
Use the generated handoff folder. Complete the Terraform and image steps in its README first.
No Studio domain or notebook GPU is required. Closing this notebook does not stop a submitted job.
The separate SageMaker execution declaration preserves the original numerical protocol.


## 1. Load the handoff
The control environment needs Python 3.12 and boto3 (`pip install -r requirements-control.txt`).
Use `aws login --profile apexpinn` outside this notebook. Never enter credentials in notebook cells.


In [ ]:
import json
import sys
from pathlib import Path

HANDOFF = Path.cwd()  # Folder containing launch_spec.json and control/.
sys.path.insert(0, str(HANDOFF / 'control'))
import sagemaker_control as control

spec = json.loads((HANDOFF / 'launch_spec.json').read_text())
infra = json.loads((HANDOFF / 'infra.json').read_text())
print({'selection': spec['selection'], 'region': spec['region'],
       'compute_status': 'NOT RUN', 'max_hours': spec['max_runtime_seconds'] / 3600})


## 2. Verify account and select the immutable image
Set IMAGE_URI to the ECR `repository@sha256:...` printed after the image push. Keep RUN_ID
unchanged when resuming; use a new JOB_NAME attempt only after the prior job is Failed/Stopped.
The default 24-hour runtime bound is a timeout, not a spending cap. Review credits and costs first.


In [ ]:
import boto3

PROFILE = 'apexpinn'  # Use None with an existing AWS execution-role session.
IMAGE_URI = 'REPLACE_WITH_ECR_REPOSITORY@sha256:DIGEST'
RUN_ID = 'apexpinn-m2-f6-mid-v1'
JOB_NAME = 'apexpinn-m2-f6-mid-v1-a01'
RESUME = False
session = boto3.Session(profile_name=PROFILE, region_name=spec['region'])
s3, sm = session.client('s3'), session.client('sagemaker')
logs = session.client('logs')
control.verify_account(session.client('sts'), s3, infra,
                       region=spec['region'], account=spec['account_id'])
request = control.build_request(spec, infra, image_uri=IMAGE_URI, job_name=JOB_NAME, run_id=RUN_ID)
print(json.dumps(request, indent=2))


## 3. Upload the exact input ZIP
This is a separate, explicit S3 write. It verifies the local archive hash and the uploaded native
S3 SHA-256 checksum, and refuses to replace an existing different object. No GPU starts here.


In [ ]:
UPLOAD_INPUT = False
if UPLOAD_INPUT:
    print(control.upload_input(s3, infra['bucket_name'], HANDOFF / spec['bundle_name'], spec))
else:
    print('NOT UPLOADED by this cell. Set UPLOAD_INPUT=True when ready.')


## 4. Submit the paid remote job
This is the only launch cell. It runs a small, remote M1/M2 checkpoint restart oracle before
M2/F6/mid/main: 20 tuning trials, five calibration fits and five final fits, then evaluation and
physics diagnostics. F1-F5 are not selected. The oracle is separate validation compute.
No automatic retries, Spot instances, warm pool or inference endpoint are requested.


In [ ]:
SUBMIT_TRAINING = False
if SUBMIT_TRAINING:
    print(control.submit(sm, s3, request, bucket=infra['bucket_name'], resume=RESUME))
else:
    print('NOT RUN. Set SUBMIT_TRAINING=True only when you choose to start paid training.')


## 5. Monitor without relying on the cell timer
Rerun these read-only cells after reconnecting. AWS reports job state and recorded runtime.
CloudWatch carries stdout/stderr. The S3 heartbeat normally refreshes about every minute plus
SageMaker sync delay; it proves the wrapper is alive, not that optimizer steps are advancing.
Inspect fit progress.json files for actual step movement. Do not relaunch to refresh the display.


In [ ]:
print(json.dumps(control.job_status(sm, JOB_NAME), indent=2, default=str))
try:
    print(json.dumps(control.heartbeat(s3, infra['bucket_name'], RUN_ID), indent=2))
except s3.exceptions.NoSuchKey:
    print('No synced heartbeat yet; check the AWS job state and CloudWatch logs.')


In [ ]:
LOG_TOKEN = globals().get('LOG_TOKEN')
page = control.recent_logs(logs, JOB_NAME, next_token=LOG_TOKEN)
for event in page.get('events', []):
    print(event['message'], end='' if event['message'].endswith('\n') else '\n')
LOG_TOKEN = page.get('nextToken')


## 6. Results and explicit recovery
After Completed, ModelArtifacts contains the S3 model.tar.gz URI. It contains the versioned
run ZIP and its receipt. Raw checkpoints, immutable attempt logs, source snapshots and reports
also persist under `runs/RUN_ID/checkpoints/`. Download and verify the ZIP/receipt before scientific
acceptance. G2 and P2 are not established by a single F6 shard.

For Failed/Stopped, inspect the original CloudWatch traceback. To resume compatible state, keep
RUN_ID, bundle hash and image digest unchanged, choose a new JOB_NAME, set RESUME=True and review
the new request. S3 sync can leave incomplete state after an abrupt loss; the runner fails closed
on missing/corrupt committed files. Uncheckpointed lost time is Unknown. Forecast evaluation is
not checkpointed per origin and may replay if interrupted. Never relabel a fresh fit as a resume.

Stopping is separate from monitoring: use AWS Console Stop training job if needed. Completed
training compute is released; S3, ECR and CloudWatch storage continue to incur their own charges.
